# PRESTO encoder, data extraction and PRESTO embedding calculation scipt

Imports & Local Working Directory Setup

In [1]:
import os
from pathlib import Path
import ee
import numpy as np
import pandas as pd
import torch
import presto

Monthly time Series extraction from GEE

In [ ]:
# Initialize Google Earth Engine
try:
  ee.Initialize()
except Exception:
  ee.Authenticate()
  ee.Initialize(project="thesis-phenology")

# Select Study Folder & Coordinate Base
output_dir = Path("output")
subfolders = sorted([f for f in output_dir.iterdir() if f.is_dir()])
if not subfolders:
  raise FileNotFoundError(f"No subdirectories found in '{output_dir}'.")

# Select active study folder (e.g., germany_barw_bbch51_...)
selected_folder = subfolders[1]
print(f"Target folder: {selected_folder.name}")

csv_files = sorted(selected_folder.glob("*modeling*.csv"))
if not csv_files:
  raise FileNotFoundError(f"No base modeling CSV found in {selected_folder.name}")

df_target = pd.read_csv(csv_files[0], sep=";")
df_target["Year"] = df_target["Year"].astype(int)

# Presto works on Sentinel-2 era (2017+)
available_years = [y for y in sorted(df_target["Year"].unique()) if y >= 2017]
print(f"Extracting inputs for years: {available_years}")

# Monthly Compositing Logic in GEE
# Presto's exact expected S2 band order
s2_bands = ["B2", "B3", "B4", "B8", "B5", "B6", "B7", "B8A", "B11", "B12"]
s1_bands = ["VV", "VH"]

dem = ee.Image("NASA/NASADEM_HGT/001").select("elevation")

def get_monthly_composite(year, month):
  """Builds a single image containing S2, S1, and ERA5 for a given month."""
  m_str = f"{month:02d}"
  start = f"{year}-{m_str}-01"
  end = (
      f"{year + 1}-01-01"
      if month == 12
      else f"{year}-{month + 1:02d}-01"
  )

  # Sentinel-2 L2A (SR) - scale to reflectances [0, 1]
  s2 = (
      ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterDate(start, end)
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 50))
      .select(s2_bands)
      .median()
  )

  # Sentinel-1 GRD (IW, descending/ascending) - linear backscatter scale
  s1 = (
      ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterDate(start, end)
      .filter(
          ee.Filter.listContains("transmitterReceiverPolarisation", "VV")
      )
      .filter(
          ee.Filter.listContains("transmitterReceiverPolarisation", "VH")
      )
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .select(s1_bands)
      .median()
  )

  # ERA5 Monthly aggregates
  era5 = (
      ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
      .filterDate(start, end)
      .select(
          ["temperature_2m", "total_precipitation_sum"],
          ["era5_temp", "era5_precip"],
      )
      .mean()
  )

  composite = (
      s2.addBands(s1)
      .addBands(era5)
      .addBands(dem.rename("elevation"))
  )
  return composite


# Batch Sampling per Year
all_year_inputs = []

for year in available_years:
  df_year = df_target[df_target["Year"] == year].copy()
  if df_year.empty:
    continue

  print(f"\nGathering monthly satellite rasters for {year}...")

  # Stack 12 months (12 x 15 bands = 180 total channels)
  monthly_bands = []
  band_names = []
  for m in range(1, 13):
    comp = get_monthly_composite(year, m)
    prefix = f"m{m:02d}_"
    # S2 (10), S1 (2), ERA5 (2), DEM (1) = 15 channels
    channels = (
        [f"s2_{b}" for b in s2_bands]
        + ["s1_VV", "s1_VH"]
        + ["temp", "precip", "elev"]
    )
    b_names = [f"{prefix}{c}" for c in channels]
    monthly_bands.append(comp)
    band_names.extend(b_names)

  year_stack = ee.Image.cat(monthly_bands).rename(band_names)

  # Prepare coordinates
  features = []
  for idx, row in df_year.iterrows():
    pt = ee.Geometry.Point([row["Longitude"], row["Latitude"]])
    features.append(ee.Feature(pt, {"row_id": int(idx)}))

  pts_fc = ee.FeatureCollection(features)

  # Batch query GEE (500 pts per chunk)
  batch_size = 500
  feature_list = pts_fc.toList(pts_fc.size())
  num_batches = int(np.ceil(len(features) / batch_size))

  sampled_records = []
  for b in range(num_batches):
    sub_fc = ee.FeatureCollection(
        feature_list.slice(b * batch_size, (b + 1) * batch_size)
    )
    sampled = year_stack.reduceRegions(
        collection=sub_fc, reducer=ee.Reducer.first(), scale=10
    )
    for feat in sampled.getInfo()["features"]:
      sampled_records.append(feat["properties"])

  df_sampled = pd.DataFrame(sampled_records).set_index("row_id")
  df_year_merged = df_year.join(df_sampled)
  all_year_inputs.append(df_year_merged)
  print(f"Extraction complete for year {year}: {len(df_year_merged)} sites.")

df_raw_satellite_inputs = pd.concat(all_year_inputs, axis=0)
raw_input_path = selected_folder / "presto_raw_inputs_temp.parquet"
df_raw_satellite_inputs.to_parquet(raw_input_path)
print(f"\nRaw satellite inputs cached to: {raw_input_path.resolve()}")

Target folder: germany_barw_bbch51_2017-01-01_2025-12-31
Extracting inputs for years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Gathering monthly satellite rasters for 2017...
Extraction complete for year 2017: 621 sites.

Gathering monthly satellite rasters for 2018...
Extraction complete for year 2018: 598 sites.

Gathering monthly satellite rasters for 2019...
Extraction complete for year 2019: 592 sites.

Gathering monthly satellite rasters for 2020...
Extraction complete for year 2020: 579 sites.

Gathering monthly satellite rasters for 2021...
Extraction complete for year 2021: 543 sites.

Gathering monthly satellite rasters for 2022...
Extraction complete for year 2022: 529 sites.

Gathering monthly satellite rasters for 2023...
Extraction complete for year 2023: 511 sites.

Gathering monthly satellite rasters for 2024...
Extraction complete for year 2024: 488 sites.

Gatherin

Load raw data, Weights & generate PRESTO embeddings

In [4]:
# Load Presto Pretrained Model on GPU / CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading Presto foundation model on {device}...")
model = presto.Presto.load_pretrained().to(device)
model.eval()

# Load Raw Inputs & Construct Presto Tensors
if "selected_folder" not in locals():
  output_dir = Path("output")
  subfolders = sorted([f for f in output_dir.iterdir() if f.is_dir()])
  selected_folder = subfolders[1]

raw_input_path = selected_folder / "presto_raw_inputs_temp.parquet"
df_inputs = pd.read_parquet(raw_input_path)
N = len(df_inputs)

print(
    f"Constructing Presto tensor inputs for {N} observations across 12 monthly steps..."
)

s2_band_keys = ["B2", "B3", "B4", "B8", "B5", "B6", "B7", "B8A", "B11", "B12"]
s1_band_keys = ["VV", "VH"]
era5_band_keys = ["temperature_2m", "total_precipitation"]
srtm_band_keys = ["elevation", "slope"]

all_x, all_mask, all_dw = [], [], []

for idx, row in df_inputs.iterrows():
  # S2 array: [timesteps=12, bands=10]
  s2_vals = np.zeros((12, 10), dtype=np.float32)
  for m in range(1, 13):
    prefix = f"m{m:02d}_"
    for b_idx, b in enumerate(s2_band_keys):
      val = row.get(f"{prefix}s2_{b}", 0.0)
      s2_vals[m - 1, b_idx] = 0.0 if pd.isna(val) else float(val)

  # S1 array: [timesteps=12, bands=2]
  s1_vals = np.zeros((12, 2), dtype=np.float32)
  for m in range(1, 13):
    prefix = f"m{m:02d}_"
    for b_idx, b in enumerate(s1_band_keys):
      val = row.get(f"{prefix}s1_{b}", -15.0)
      s1_vals[m - 1, b_idx] = -15.0 if pd.isna(val) else float(val)

  # ERA5 array: [timesteps=12, bands=2]
  era5_vals = np.zeros((12, 2), dtype=np.float32)
  for m in range(1, 13):
    prefix = f"m{m:02d}_"
    t_val = row.get(f"{prefix}temp", 283.15)
    p_val = row.get(f"{prefix}precip", 0.0)
    era5_vals[m - 1, 0] = 283.15 if pd.isna(t_val) else float(t_val)
    era5_vals[m - 1, 1] = 0.0 if pd.isna(p_val) else float(p_val)

  # SRTM array: [timesteps=12, bands=2]
  elev_val = row.get("m01_elev", 100.0)
  elev = 100.0 if pd.isna(elev_val) else float(elev_val)
  srtm_vals = np.tile(np.array([elev, 0.0], dtype=np.float32), (12, 1))

  x, mask, dw = presto.construct_single_presto_input(
      s2=torch.tensor(s2_vals, dtype=torch.float32),
      s2_bands=s2_band_keys,
      s1=torch.tensor(s1_vals, dtype=torch.float32),
      s1_bands=s1_band_keys,
      era5=torch.tensor(era5_vals, dtype=torch.float32),
      era5_bands=era5_band_keys,
      srtm=torch.tensor(srtm_vals, dtype=torch.float32),
      srtm_bands=srtm_band_keys,
  )
  all_x.append(x)
  all_mask.append(mask)
  all_dw.append(dw)

# Stack into batched tensors
x_tensor = torch.stack(all_x)
mask_tensor = torch.stack(all_mask)
dw_tensor = torch.stack(all_dw).long()
latlon_tensor = torch.tensor(
    df_inputs[["Latitude", "Longitude"]].values.copy(), dtype=torch.float32
)


# Batched PRESTO Embedding Extraction
batch_size = 256
num_batches = int(np.ceil(N / batch_size))

all_embeddings = []

print(f"Running PRESTO encoder across {N} observations...")

with torch.no_grad():

    for b in range(num_batches):

        start = b * batch_size
        end = min((b + 1) * batch_size, N)

        bx = x_tensor[start:end].float().to(device)
        bmask = mask_tensor[start:end].to(device)
        bdw = dw_tensor[start:end].long().to(device)
        blatlon = latlon_tensor[start:end].float().to(device)

        # Request globally pooled embeddings directly
        embeddings = model.encoder(
            bx,
            dynamic_world=bdw,
            mask=bmask,
            latlons=blatlon,
            month=0,
            eval_task=True
        )

        # Inspect and validate the returned embeddings
        if isinstance(embeddings, tuple):
            embeddings = embeddings[0]

        print(
            f"Batch {b+1}/{num_batches}: "
            f"Output shape = {embeddings.shape}"
        )

        # Expected output: [batch_size, 128]
        if embeddings.ndim != 2 or embeddings.shape[-1] != 128:
            raise RuntimeError(
                f"Unexpected PRESTO embedding shape: "
                f"{embeddings.shape}. Expected [B, 128]."
            )

        all_embeddings.append(
            embeddings.cpu().numpy()
        )

# Combine all batches
embedding_matrix = np.concatenate(all_embeddings, axis=0)

print(
    "Successfully generated PRESTO embeddings:",
    embedding_matrix.shape
)

assert embedding_matrix.shape == (N, 128), (
    f"Expected {(N, 128)}, got {embedding_matrix.shape}"
)


# Attach Embeddings & Save Curated Dataset
embedding_cols = [f"presto_{i}" for i in range(128)]
df_presto_features = pd.DataFrame(
    embedding_matrix, columns=embedding_cols, index=df_inputs.index
)

base_cols = [
    c
    for c in df_inputs.columns
    if not (c.startswith("m0") or c.startswith("m1"))
]
df_final = pd.concat([df_inputs[base_cols], df_presto_features], axis=1)

output_csv_path = (
    selected_folder / "curated_for_modeling_with_presto_embeddings.csv"
)
df_final.to_csv(output_csv_path, sep=";", index=False)

print(
    f"\nSuccessfully generated {output_csv_path.name} with 128 Presto"
    " dimensions!"
)
print(f"Saved to: {output_csv_path.resolve()}")

Loading Presto foundation model on cpu...


Constructing Presto tensor inputs for 4942 observations across 12 monthly steps...
Running PRESTO encoder across 4942 observations...
Batch 1/20: Output shape = torch.Size([256, 128])
Batch 2/20: Output shape = torch.Size([256, 128])
Batch 3/20: Output shape = torch.Size([256, 128])
Batch 4/20: Output shape = torch.Size([256, 128])
Batch 5/20: Output shape = torch.Size([256, 128])
Batch 6/20: Output shape = torch.Size([256, 128])
Batch 7/20: Output shape = torch.Size([256, 128])
Batch 8/20: Output shape = torch.Size([256, 128])
Batch 9/20: Output shape = torch.Size([256, 128])
Batch 10/20: Output shape = torch.Size([256, 128])
Batch 11/20: Output shape = torch.Size([256, 128])
Batch 12/20: Output shape = torch.Size([256, 128])
Batch 13/20: Output shape = torch.Size([256, 128])
Batch 14/20: Output shape = torch.Size([256, 128])
Batch 15/20: Output shape = torch.Size([256, 128])
Batch 16/20: Output shape = torch.Size([256, 128])
Batch 17/20: Output shape = torch.Size([256, 128])
Batch 18